# Overview

This notebook provides an exploratory view of hospital length of stay using the UCI Diabetes 130-US Hospitals dataset. The goal is to understand the target, inspect data quality, and surface practical business patterns that may affect inpatient stay duration.

# Data source & business context

The data captures inpatient encounters for diabetic patients across U.S. hospitals. For hospital operations, `time_in_hospital` is a useful outcome because it relates to bed utilization, care coordination, discharge planning, and overall resource demand. This notebook stays focused on exploration and business understanding rather than model training.

# Load data

In [ ]:
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    notebook_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    notebook_dir = os.getcwd()

candidate_roots = [
    notebook_dir,
    os.path.abspath(os.path.join(notebook_dir, '..')),
]

repo_root = None
for candidate in candidate_roots:
    candidate_data_dir = os.path.join(candidate, 'data', 'raw')
    candidate_data_path = os.path.join(candidate_data_dir, 'diabetic_data.csv')
    candidate_mapping_path = os.path.join(candidate_data_dir, 'IDS_mapping.csv')
    if os.path.exists(candidate_data_path) and os.path.exists(candidate_mapping_path):
        repo_root = candidate
        break

if repo_root is None:
    repo_root = os.path.abspath(os.path.join(notebook_dir, '..')) if os.path.basename(notebook_dir) == 'notebooks' else notebook_dir

DATA_DIR = os.path.join(repo_root, 'data', 'raw')
DATA_PATH = os.path.join(DATA_DIR, 'diabetic_data.csv')
MAPPING_PATH = os.path.join(DATA_DIR, 'IDS_mapping.csv')

missing_files = [path for path in [DATA_PATH, MAPPING_PATH] if not os.path.exists(path)]
if missing_files:
    missing_list = '\n'.join(missing_files)
    raise FileNotFoundError(
        'Required data files were not found. Place diabetic_data.csv and IDS_mapping.csv in data/raw/. Missing files:\n'
        f'{missing_list}'
    )

df = pd.read_csv(DATA_PATH)
ids_map = pd.read_csv(MAPPING_PATH)
df_inspect = df.replace('?', np.nan)

print(f'df shape: {df.shape}')
print(f'ids_map shape: {ids_map.shape}')
print('\ndf columns:')
print(df.columns.tolist())
print('\nids_map columns:')
print(ids_map.columns.tolist())

display(df.head())
display(ids_map.head())


# Data dictionary

The table below highlights core fields that are most relevant for understanding and eventually predicting length of stay.

# Missing data and data quality

Before analyzing the target, it is helpful to inspect missingness and placeholder values. In this dataset, `?` is commonly used as a stand-in for missing values, so the notebook converts `?` to `NaN` for inspection summaries.

# Target analysis: time_in_hospital

This section evaluates the distribution of `time_in_hospital`, summarizes its basic statistics, and compares average length of stay across patient and encounter categories.

In [ ]:
target_col = 'time_in_hospital'
if target_col not in df_inspect.columns:
    raise KeyError("Column 'time_in_hospital' was not found in diabetic_data.csv")

target = pd.to_numeric(df_inspect[target_col], errors='coerce')

print('Descriptive statistics for time_in_hospital:')
print(target.describe())
print()
print(f"Mean: {target.mean():.2f}")
print(f"Median: {target.median():.2f}")
print(f"Min: {target.min()}")
print(f"Max: {target.max()}")

print('\nMissing-values summary after replacing ? with NaN:')
missing_summary = (
    df_inspect.isna()
    .sum()
    .rename('missing_count')
    .to_frame()
    .assign(missing_pct=lambda x: (x['missing_count'] / len(df_inspect) * 100).round(2))
    .sort_values(['missing_count', 'missing_pct'], ascending=False)
)
display(missing_summary.head(15))

main_fields = [
    ('encounter_id', 'Unique encounter identifier', 'High-cardinality ID; not a business driver itself'),
    ('patient_nbr', 'Unique patient identifier', 'Useful for repeat-visit logic and leakage checks'),
    ('race', 'Patient race category', 'Demographic segment that may reflect access and care pattern differences'),
    ('gender', 'Patient gender', 'Demographic grouping for LOS comparison'),
    ('age', 'Age band of the patient', 'Broad clinical risk and utilization signal'),
    ('admission_type_id', 'Type of admission', 'Often linked to urgency and severity at intake'),
    ('discharge_disposition_id', 'Discharge destination or status', 'May reflect complexity of care and recovery path'),
    ('admission_source_id', 'Where the admission came from', 'Captures referral or emergency intake patterns'),
    ('num_lab_procedures', 'Number of lab procedures', 'Proxy for clinical workup intensity'),
    ('num_medications', 'Number of medications', 'Proxy for treatment complexity'),
    ('number_inpatient', 'Prior inpatient visits', 'Possible signal of acuity or chronic complexity'),
    ('diag_1', 'Primary diagnosis code', 'Important clinical context, but requires cleaning/grouping'),
    ('time_in_hospital', 'Days spent in the hospital', 'Target variable for LOS analysis')
]
data_dictionary = pd.DataFrame(main_fields, columns=['field', 'business_description', 'why_it_matters'])
print('\nSmall data dictionary for main modeling fields:')
display(data_dictionary)

group_fields = [
    'age',
    'gender',
    'race',
    'admission_type_id',
    'discharge_disposition_id',
    'admission_source_id',
]

print('\nGrouped summaries of time_in_hospital:')
for field in group_fields:
    field_series = df_inspect[field].fillna('Missing')
    summary = (
        pd.DataFrame({field: field_series, target_col: target})
        .groupby(field, dropna=False)[target_col]
        .agg(['count', 'mean', 'median'])
        .sort_values(['mean', 'count'], ascending=[False, False])
        .round(2)
    )
    print(f'\n{field}')
    display(summary.head(15))

plot_df = df_inspect.copy()
plot_df[target_col] = target

age_order = sorted(
    [value for value in plot_df['age'].dropna().unique()],
    key=lambda x: int(str(x).strip('[]()').split('-')[0])
)
age_mean = plot_df.groupby('age', dropna=False)[target_col].mean().reindex(age_order)

gender_mean = (
    plot_df.assign(gender=plot_df['gender'].fillna('Missing'))
    .groupby('gender')[target_col]
    .mean()
    .sort_values(ascending=False)
)
race_top = plot_df['race'].fillna('Missing').value_counts().head(8).index
race_mean = plot_df.loc[plot_df['race'].fillna('Missing').isin(race_top)].copy()
race_mean['race'] = race_mean['race'].fillna('Missing')
race_mean = race_mean.groupby('race')[target_col].mean().sort_values(ascending=False)

admission_type_mean = (
    plot_df.groupby('admission_type_id')[target_col]
    .mean()
    .sort_values(ascending=False)
    .head(10)
)
discharge_mean = (
    plot_df.groupby('discharge_disposition_id')[target_col]
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

fig, axes = plt.subplots(3, 2, figsize=(16, 18))
axes = axes.flatten()

axes[0].hist(target.dropna(), bins=range(int(target.min()), int(target.max()) + 2), edgecolor='black', alpha=0.85, color='#4C78A8')
axes[0].set_title('Distribution of time_in_hospital')
axes[0].set_xlabel('Days in hospital')
axes[0].set_ylabel('Frequency')

axes[1].boxplot(target.dropna(), vert=True)
axes[1].set_title('Spread of LOS values')
axes[1].set_ylabel('Days in hospital')
axes[1].set_xticks([1])
axes[1].set_xticklabels(['time_in_hospital'])

axes[2].bar(range(len(age_mean)), age_mean.values, color='#F58518')
axes[2].set_title('Average LOS by age band')
axes[2].set_xlabel('Age band')
axes[2].set_ylabel('Mean days')
axes[2].set_xticks(range(len(age_mean)))
axes[2].set_xticklabels(age_mean.index, rotation=45, ha='right')

axes[3].bar(gender_mean.index.astype(str), gender_mean.values, color='#54A24B')
axes[3].set_title('Average LOS by gender')
axes[3].set_xlabel('Gender')
axes[3].set_ylabel('Mean days')

axes[4].bar(race_mean.index.astype(str), race_mean.values, color='#E45756')
axes[4].set_title('Average LOS by race (top groups)')
axes[4].set_xlabel('Race')
axes[4].set_ylabel('Mean days')
axes[4].tick_params(axis='x', rotation=45)

axes[5].bar(admission_type_mean.index.astype(str), admission_type_mean.values, color='#72B7B2', alpha=0.9)
axes[5].set_title('Average LOS by admission type ID')
axes[5].set_xlabel('Category ID')
axes[5].set_ylabel('Mean days')
axes[5].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


# Feature exploration

The grouped summaries and charts help identify which demographic and encounter attributes appear to move length of stay. Fields with strong differences in average LOS are strong candidates for deeper business review.

# Correlates of length of stay

From an operational perspective, admission pathway variables, discharge outcomes, age bands, and indicators of treatment complexity are likely to be among the most important correlates of inpatient stay duration.

# Modeling plan

A later modeling workflow should translate these exploratory findings into cleaned features, encoded categorical variables, leakage checks, and carefully defined train-validation splits. That work is intentionally out of scope for this notebook.

# Limitations

This exploratory pass depends on source data quality and coded hospital metadata. Several business-important IDs require mapping and interpretation, and summary statistics alone do not establish causal drivers of length of stay.

# Final summary

Length of stay matters operationally because it affects bed availability, discharge planning, staffing pressure, and overall hospital throughput. Better LOS estimates can help care teams and hospital leaders plan ahead and respond earlier when patients are likely to need longer stays.

In this exploratory review, longer stays appear to be more closely associated with factors tied to patient complexity and encounter pathway, including age band, admission type, discharge disposition, admission source, medication burden, and prior inpatient utilization. These patterns are useful because they point to signals that may help explain why some encounters require more hospital time than others.

The modeling pipeline will aim to predict `time_in_hospital`, measured in days, using the encounter and patient features available at or around admission. The goal is to support business understanding and operational planning rather than replace clinical judgment.